# Baseline Model

*Research Question: Can publicly available socioeconomic and geographic data approximate language endangerment status without requiring on-the-ground linguistic assessment?*

This notebook establishes a baseline for subsequent models using a majority class classifier and logistic regression for multi-class classification on the ELCAT data. The task is to predict endangerment_level with the available socioeconomic and geographic features.

**Results**

- Majority class model accuracy: **27.73%**
- Logistic regression model accuracy: 

In [75]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn import preprocessing

In [51]:
df = pd.read_csv('../data/elcat_master_df.csv')

### Data preprocessing

- Log transform GDP
- Drop the languages that have null values for World Bank indicators since logistic regression does not natively handle NaNs.
- Select Columns for model 
    - Drop domains_of_use, speaker_number, speaker_number_trends, transmission
    - Use only averages instead of max for gdp, internet, urban growth and urban population
    - Drop lang_family since it is too sparse for logistic regression. We can build this into the ensemble model later on to see if it impacts the predictions.
- Shuffled Dataset
- Create train, val and test sets with 70/15/15 split
- One-hot encode Y values

In [52]:
df['log_avg_gdp_percapita'] = np.log(df['avg_gdp_percapita'])
df = df.drop(columns=['avg_gdp_percapita'], axis=1)

In [53]:
df = df.drop(df[df['log_avg_gdp_percapita'].isna()].index, axis=0)

In [54]:
np.random.seed(12)

indices = np.array([i for i in range(len(df))])

shuffled_indices = np.random.permutation(indices)
df = df.iloc[shuffled_indices].reset_index(drop=True)

In [55]:
Y = df['elcat_endangerment']
X = df[['lang_family_count','country_count','log_avg_gdp_percapita','avg_internet','avg_urban','avg_urban_growth','official',
       'regional','national','minority','widely_spoken','area_Africa','area_Australia','area_Eurasia','area_North America',
       'area_Papunesia','area_South America']]

In [56]:
X_temp, X_test, Y_temp, Y_test = train_test_split(X, Y, test_size = 0.15, random_state = 12)
X_train, X_val, Y_train, Y_val = train_test_split(X_temp, Y_temp, test_size = 0.15, random_state = 12)

In [ ]:
label_encoder = preprocessing.LabelEncoder()
Y_train_encoded = label_encoder.fit_transform(Y_train)
Y_val_encoded = label_encoder.fit_transform(Y_val)
Y_test_encoded = label_encoder.fit_transform(Y_test)

Y_train_dense = 

### Majority Class Model

This is a simple model that predicts the majority class for every sample. 

In [66]:
majority_class = np.bincount(Y_train_encoded).argmax()

majority_prediction = [majority_class] * len(Y_train_encoded)

majority_accuracy = accuracy_score(Y_train_encoded, majority_prediction)
print(f'Majority class model accuracy: {majority_accuracy*100:.2f}%')

Majority class model accuracy: 27.73%


In [73]:
# Log loss calculation
loss = log_loss(Y_train_encoded, majority_prediction)
print(f'Majority class model log loss: {loss:.4f}')

ValueError: y_true and y_pred contain different number of classes 8, 2. Please provide the true labels explicitly through the labels argument. Classes found in y_true: [0 1 2 3 4 5 6 7]

### Logistic Regression Model